# NB02 — Build the training corpus

**Project:** CardioMamba-Net · **Stage:** 2 of 5
`01_verify_and_download` → **`02_preprocess_to_hf`** → `03_baselines` → `04_cardiomamba_train` → `05_evaluate_and_figures`

---

## What this does

NB01 proved the mirror is real and left a **decimated 128 Hz copy of the whole corpus** on
Hugging Face. This notebook turns that into a training corpus: it adds the supervision targets,
assigns folds, and computes normalisation statistics — then publishes the result so NB03–NB05
never touch raw data again.

| Step | Why |
|---|---|
| Pull `cr-rvs-radar-ecg-inventory` from HF | The 8 physics channels at 128 Hz already exist; re-deriving them would be wasted work |
| Quality filter on `quality_flags` + `beat_coupling` | Exclude on evidence, not vibes. A recording where the radar demonstrably cannot see the heartbeat teaches the model nothing but noise |
| Build ECG target, R-peak heatmap, instantaneous RR | The three heads of C4 need three targets |
| Window index at 50 % overlap, flagged for no-overlap | One index serves both train (overlapped) and test (not) — see the leakage note below |
| Subject-wise 5-fold **and** LOSO fold assignment | Experiments A/B/C use the folds; Experiment D uses LOSO |
| Per-fold normalisation stats, **train windows only** | Computing them corpus-wide is invisible leakage |

## ⚠️ Accelerator: **None (CPU)**

This notebook is disk- and CPU-bound — it decodes, filters and windows arrays. A GPU would sit
idle while burning your weekly T4 quota, which NB03 and NB04 genuinely need. Set
*Session options → Accelerator → **None***.

## The leakage decision, made explicit

The baseline overlaps windows by 50 % and reports Table 1 counts that already carry the overlap,
then splits those totals 80/20. NB01 confirmed the arithmetic: our overlapped counts reproduce
theirs to within 0.1 % (Apnea matched exactly, 1140 = 1140). That means **overlapping windows can
straddle train and test**, and adjacent windows share 512 of their 1024 samples.

We do not repeat that. Here:

- **Splits are always by subject** — no subject appears in more than one split, ever.
- **Train windows overlap 50 %** (data augmentation, as intended).
- **Validation and test windows do not overlap at all** — the `no_overlap` column selects them.

That is stricter than the baseline, and it means our numbers are, if anything, pessimistic
relative to theirs. We say so in the paper.

---

## How to run

1. *Session options* → **Accelerator: None**, **Internet: On**.
2. *Add-ons → Secrets* → `HF_TOKEN` (write) attached.
3. Run all. Expect **20–45 minutes**, most of it the download and the final upload.
4. Interrupt-safe and resumable, same contract as NB01.

---
# 1 · Configuration

In [ ]:
CFG = {
    "SRC_REPO":   "Shanmuk4622/cr-rvs-radar-ecg-inventory",     # NB01 output
    "DST_REPO":   "Shanmuk4622/cr-rvs-radar-ecg-processed",     # this notebook's output
    "HF_PRIVATE": False,
    "RUN_ID":     "nb02_corpus_v1",

    "WORK":    "/kaggle/working/nb02",
    "SCRATCH": "/kaggle/temp/nb02",

    "PUSH_INTERVAL_S": 30 * 60,
    "HF_MAX_REQ_HOUR": 120,

    # frozen to Chowdhury et al. 2024 section 2.3
    "FS": 128, "WINDOW": 1024, "HOP_TRAIN": 512,
    "PEAK_SIGMA": 3.0,          # samples; ~23 ms at 128 Hz

    # quality gates -- exclusion is evidence-based, and every exclusion is logged
    "MIN_BEAT_COUPLING": 1.30,  # radar must demonstrably see the heartbeat
    "MIN_DURATION_S":    60.0,
    "EXCLUDE_FLAGS":     ["NO_RADAR_CHANNELS", "missing_channels", "ecg_flatline",
                          "few_or_no_rpeaks", "implausible_hr", "nan_in_radar"],
    "WARN_ONLY_FLAGS":   ["iq_clipping", "strong_iq_imbalance", "large_sync_lag",
                          "very_short", "radar_ecg_length_mismatch"],

    "N_FOLDS": 5,
    "SEED": 1337,
    "SMOKE_TEST": False,
}
import json
print(json.dumps(CFG, indent=2))

In [ ]:
import os, sys, gc, re, json, math, time, signal, atexit, threading, warnings, subprocess, platform
from pathlib import Path
from datetime import datetime, timezone
warnings.filterwarnings("ignore")

def _pip(*p):
    miss = [x for x in p if __import__("importlib").util.find_spec(x.replace("-", "_")) is None]
    if miss:
        print("installing:", miss)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *miss], check=False)
_pip("pyarrow", "huggingface_hub")

import numpy as np, pandas as pd
from scipy import signal as ss

WORK = Path(CFG["WORK"]); SCRATCH = Path(CFG["SCRATCH"])
for d in (WORK, SCRATCH, WORK / "recordings", WORK / "logs"):
    d.mkdir(parents=True, exist_ok=True)
np.random.seed(CFG["SEED"])

def disk(p):
    try:
        s = os.statvfs(p)
        return f"{s.f_bavail*s.f_frsize/2**30:.1f} GB free"
    except Exception:
        return "?"
print("working:", disk("/kaggle/working"), " temp:",
      disk("/kaggle/temp") if Path("/kaggle/temp").exists() else "absent")
print("numpy", np.__version__, "| pandas", pd.__version__)

---
# 2 · The shared library

Three modules are written to disk here and **re-used verbatim by NB03, NB04 and NB05**:
`crvs_sync.py` (the HF cadence rules), `crvs_data.py` (windowing, folds, the Dataset) and
`crvs_metrics.py` (every metric). Defining them once means every experiment sees byte-identical
inputs and every number is computed the same way.

They are also pushed to the processed-data repo, so the later notebooks download them rather
than carrying a divergent copy.

In [ ]:
CRVS_SYNC_SRC = r"""
# crvs_sync.py -- resumable, rate-limited, interrupt-safe Hugging Face folder sync.
# Identical across NB01-NB05 so the cadence rules are enforced in exactly one place.
#   * push at most once per PUSH_INTERVAL_S (default 30 min)
#   * push immediately when a stage finishes            -> sync.stage_done("name")
#   * push immediately when execution is stopped        -> SIGINT / SIGTERM / atexit
#   * one upload_folder call per flush, behind a token bucket, backing off on 429
#   * resume by pulling the run folder back on startup
import os, json, time, random, threading, atexit, signal
from pathlib import Path
from datetime import datetime, timezone

class TokenBucket:
    # capacity = requests per hour, refilled continuously
    def __init__(self, per_hour=120):
        self.capacity = float(per_hour); self.tokens = float(per_hour)
        self.rate = per_hour / 3600.0; self.t = time.monotonic()
        self.lock = threading.Lock()
    def take(self, n=1, block=True, timeout=1200):
        deadline = time.monotonic() + timeout
        while True:
            with self.lock:
                now = time.monotonic()
                self.tokens = min(self.capacity, self.tokens + (now - self.t) * self.rate)
                self.t = now
                if self.tokens >= n:
                    self.tokens -= n; return True
                need = (n - self.tokens) / self.rate
            if not block or time.monotonic() + need > deadline:
                return False
            time.sleep(min(need, 5.0))

class HFSync:
    def __init__(self, repo_id, local_dir, token, repo_type="dataset", private=False,
                 run_id="run", push_interval_s=1800, max_req_hour=120, retry_max=6,
                 verbose=True):
        from huggingface_hub import HfApi
        self.api = HfApi(token=token); self.token = token
        self.repo_id = repo_id; self.repo_type = repo_type; self.private = private
        self.run_id = run_id
        self.local = Path(local_dir); self.local.mkdir(parents=True, exist_ok=True)
        self.interval = push_interval_s
        self.bucket = TokenBucket(max_req_hour)
        self.retry_max = retry_max; self.verbose = verbose
        self._last_push = 0.0
        self._flag = threading.Event(); self._stop = threading.Event()
        self._lock = threading.Lock()
        self._pushes = 0; self._failures = 0
        self.history = self.local / "history.jsonl"
        self.state_path = self.local / "state.json"
        self._ensure_repo(); self._install_handlers()
        self._thread = threading.Thread(target=self._loop, daemon=True, name="hf-uploader")
        self._thread.start()
        self.log("sync_started", repo=self.repo_id, private=self.private)

    def _ensure_repo(self):
        from huggingface_hub import create_repo
        create_repo(self.repo_id, repo_type=self.repo_type, private=self.private,
                    exist_ok=True, token=self.token)
        if not self.private:
            try:
                self.api.update_repo_visibility(self.repo_id, private=False,
                                                repo_type=self.repo_type, token=self.token)
            except Exception:
                pass

    @property
    def url(self):
        kind = "datasets/" if self.repo_type == "dataset" else ""
        return "https://huggingface.co/" + kind + self.repo_id

    def log(self, event, **kw):
        rec = {"ts": datetime.now(timezone.utc).isoformat(), "run": self.run_id, "event": event}
        rec.update(kw)
        try:
            with open(self.history, "a") as f:
                f.write(json.dumps(rec, default=str) + "\n")
        except Exception:
            pass
        if self.verbose and event not in ("heartbeat",):
            print("  [" + event + "] " + " ".join(f"{k}={v}" for k, v in kw.items()))

    def save_state(self, state):
        tmp = self.state_path.with_suffix(".tmp")
        tmp.write_text(json.dumps(state, indent=2, default=str)); tmp.replace(self.state_path)

    def load_state(self, default=None):
        if self.state_path.exists():
            try:
                return json.loads(self.state_path.read_text())
            except Exception:
                pass
        return default if default is not None else {}

    def pull(self, allow_patterns=None, into=None):
        from huggingface_hub import snapshot_download
        try:
            self.bucket.take(1)
            p = snapshot_download(self.repo_id, repo_type=self.repo_type, token=self.token,
                                  local_dir=str(into or self.local),
                                  allow_patterns=allow_patterns)
            self.log("resume_pull_ok", path=str(p)); return True
        except Exception as e:
            self.log("resume_pull_empty", err=type(e).__name__); return False

    def stage_done(self, name, **kw):
        self.log("stage_done", stage=name, **kw); self._flag.set()

    def _do_upload(self, msg):
        from huggingface_hub import upload_folder
        for attempt in range(self.retry_max):
            if not self.bucket.take(1, block=True, timeout=1800):
                self.log("rate_limited_giveup"); return False
            try:
                upload_folder(folder_path=str(self.local), repo_id=self.repo_id,
                              repo_type=self.repo_type, token=self.token,
                              commit_message=msg,
                              ignore_patterns=["*.tmp", "**/__pycache__/**", ".git*",
                                               "*.lock", ".cache/**"])
                self._pushes += 1; self._last_push = time.time()
                self.log("push_ok", n=self._pushes, msg=msg); return True
            except Exception as e:
                self._failures += 1
                wait = min(300, (2 ** attempt) * 5) * (0.7 + 0.6 * random.random())
                self.log("push_retry", attempt=attempt + 1,
                         err=f"{type(e).__name__}: {e}", sleep=round(wait, 1))
                time.sleep(wait)
        self.log("push_failed_permanently", msg=msg); return False

    def flush(self, final=False, msg=None):
        with self._lock:
            stamp = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M")
            m = msg or ((self.run_id + " final") if final else (self.run_id + " @ " + stamp + "Z"))
            ok = self._do_upload(m); self._flag.clear(); return ok

    def _loop(self):
        while not self._stop.is_set():
            self._stop.wait(20)
            if self._stop.is_set():
                break
            due = (time.time() - self._last_push) >= self.interval
            want = self._flag.is_set()
            if due or want:
                try:
                    tag = "stage" if want else "periodic"
                    self.flush(msg=self.run_id + " " + tag + " @ " +
                               datetime.now(timezone.utc).strftime("%H:%M") + "Z")
                except Exception as e:
                    self.log("loop_error", err=str(e))

    def _install_handlers(self):
        def handler(signum, frame):
            self.log("interrupt", signal=int(signum))
            try:
                self.flush(final=True, msg=self.run_id + " interrupted (sig " + str(signum) + ")")
            finally:
                if signum == signal.SIGINT:
                    raise KeyboardInterrupt
        for sig in (signal.SIGINT, signal.SIGTERM):
            try:
                signal.signal(sig, handler)
            except Exception:
                pass
        atexit.register(self.close)

    def close(self):
        if self._stop.is_set():
            return
        self.log("closing"); self._stop.set()
        try:
            self.flush(final=True)
        except Exception:
            pass
"""
CRVS_DATA_SRC = r"""
# crvs_data.py -- windowing, folds, normalisation and the torch Dataset.
# Shared by NB03, NB04 and NB05 so every experiment sees byte-identical inputs.
import json, math
import numpy as np
from pathlib import Path

CHANNELS = ["I", "Q", "phi", "dy", "vel", "acc", "amp", "cardiac"]
# One recording is stored as a single UNCOMPRESSED .npy of shape (len(ARRAY_ROWS), n).
# It has to be .npy, not .npz: np.load(..., mmap_mode="r") silently IGNORES mmap_mode on an
# .npz, so every __getitem__ would decompress all 11 arrays to slice 1024 samples out of
# each -- measured at 23 ms per window, which would dominate the GPU time on Kaggle.
ARRAY_ROWS = CHANNELS + ["ecg_norm", "peak_map", "rr_ms"]
ROW = {name: i for i, name in enumerate(ARRAY_ROWS)}
FS       = 128
# Bumped whenever this module changes in a way the notebooks depend on. Every notebook
# asserts it after import, because writing a .py and importing it is NOT idempotent inside
# one kernel: Python caches the module in sys.modules, so a second run silently keeps the
# first version. That is how a stale .npz loader survived a rebuilt notebook once already.
LIB_VERSION = 3
WINDOW   = 1024          # 8.0 s, frozen to Chowdhury et al. 2024 section 2.3.4
HOP_TRAIN = 512          # 50 % overlap on train only
SCENARIOS = ["Resting", "Valsalva", "Apnea", "Tilt-up", "Tilt-down"]

def canon_scenario(s):
    s = str(s).strip().lower()
    for key, out in [("tiltdown", "Tilt-down"), ("tilt_down", "Tilt-down"), ("tilt-down", "Tilt-down"),
                     ("tiltup", "Tilt-up"), ("tilt_up", "Tilt-up"), ("tilt-up", "Tilt-up"),
                     ("valsalva", "Valsalva"), ("apnea", "Apnea"), ("apnoea", "Apnea"),
                     ("rest", "Resting")]:
        if key in s:
            return out
    return str(s)

def range_normalise(x, eps=1e-8):
    # z-score then squash to [-1, 1]; the baseline used [0, 1], we declare the change
    x = np.asarray(x, np.float32)
    sd = float(x.std())
    if not np.isfinite(sd) or sd < eps:
        return np.zeros_like(x, np.float32)          # constant input -> 0, not -1
    x = (x - x.mean()) / (sd + eps)
    lo, hi = np.percentile(x, 0.5), np.percentile(x, 99.5)
    x = np.clip(x, lo, hi)
    rng = float(hi - lo)
    if rng < eps:
        return np.zeros_like(x, np.float32)
    return (2.0 * (x - lo) / rng - 1.0).astype(np.float32)

def peak_heatmap(n, peaks, sigma=3.0):
    # Gaussian bumps at each R peak -- the target for the multi-task peak head
    y = np.zeros(n, np.float32)
    if len(peaks) == 0:
        return y
    half = int(math.ceil(3 * sigma))
    g = np.exp(-0.5 * (np.arange(-half, half + 1) / sigma) ** 2).astype(np.float32)
    for p in np.asarray(peaks, int):
        a, b = max(0, p - half), min(n, p + half + 1)
        y[a:b] = np.maximum(y[a:b], g[a - (p - half): (b - (p - half))])
    return y

def rr_curve(n, peaks, fs=FS, lo_ms=300.0, hi_ms=2000.0):
    # per-sample instantaneous RR interval in ms, linearly interpolated between beats
    out = np.full(n, np.nan, np.float32)
    p = np.asarray(peaks, int)
    if len(p) < 3:
        return np.nan_to_num(out, nan=800.0)
    rr = np.diff(p) / fs * 1000.0
    mid = (p[:-1] + p[1:]) / 2.0
    ok = (rr > lo_ms) & (rr < hi_ms)
    if ok.sum() < 2:
        return np.nan_to_num(out, nan=float(np.median(rr)))
    out = np.interp(np.arange(n), mid[ok], rr[ok]).astype(np.float32)
    return out

_SLOW_WARNED = {"npz": False}

class _Rec:
    # Reads one recording in whichever format is on disk.
    #   .npy (preferred) -- uncompressed, genuinely memory-mapped, ~0.3 ms per window
    #   .npz (legacy)    -- what an earlier NB02 wrote; correct but ~85x slower, because
    #                       np.load ignores mmap_mode on a zip archive and every window
    #                       decompresses all 11 arrays.
    # Both are supported so an existing corpus keeps working without a 400 MB re-upload.
    __slots__ = ("data", "kind")

    def __init__(self, rec_dir, rid):
        d = Path(rec_dir)
        pnpy, pnpz = d / (rid + ".npy"), d / (rid + ".npz")
        if pnpy.exists():
            self.data = np.load(pnpy, mmap_mode="r"); self.kind = "npy"
        elif pnpz.exists():
            self.data = np.load(pnpz); self.kind = "npz"
            if not _SLOW_WARNED["npz"]:
                _SLOW_WARNED["npz"] = True
                print("  note: reading legacy .npz recordings. Correct, but about 85x slower "
                      "per window than .npy -- re-run NB02 to regenerate the corpus and cut "
                      "the data-loading cost.")
        else:
            raise FileNotFoundError(
                f"no recording for '{rid}' in {d} (looked for .npy and .npz). "
                "Either NB02 did not finish, or the snapshot_download allow_patterns in "
                "this notebook do not cover the format NB02 wrote.")

    def rows(self, names, s, e):
        if self.kind == "npy":
            return np.array(self.data[[ROW[n] for n in names], s:e], np.float32)
        return np.stack([np.array(self.data[n][s:e], np.float32) for n in names], 0)

    def one(self, name, s, e):
        if self.kind == "npy":
            return np.array(self.data[ROW[name], s:e], np.float32)
        return np.array(self.data[name][s:e], np.float32)

class WindowDataset:
    # Slices windows on the fly, so changing WINDOW or the overlap never requires
    # re-running NB02.
    def __init__(self, rec_dir, index, norm=None, channels=None, augment=False, seed=0):
        self.rec_dir = Path(rec_dir)
        self.index = index.reset_index(drop=True)
        self.norm = norm
        self.channels = channels or CHANNELS
        self.rows = [ROW[c] for c in self.channels]
        self.augment = augment
        self.rng = np.random.RandomState(seed)
        self._cache = {}

    def __len__(self):
        return len(self.index)

    def _rec(self, rid):
        if rid not in self._cache:
            if len(self._cache) > 48:
                self._cache.pop(next(iter(self._cache)))
            self._cache[rid] = _Rec(self.rec_dir, rid)
        return self._cache[rid]

    def __getitem__(self, i):
        import torch
        r = self.index.iloc[i]
        z = self._rec(r["rec_id"])
        s, e = int(r["start"]), int(r["start"]) + WINDOW
        # _Rec.rows / _Rec.one always np.array (copy), never a view into a read-only
        # memmap -- torch.from_numpy on a non-writable array is undefined behaviour.
        x = z.rows(self.channels, s, e)
        if self.norm is not None:
            mu = np.asarray(self.norm["mean"], np.float32)[:, None]
            sd = np.asarray(self.norm["std"], np.float32)[:, None]
            x = (x - mu) / (sd + 1e-6)
        x = np.clip(x, -8.0, 8.0)
        y  = z.one("ecg_norm", s, e)
        pk = z.one("peak_map", s, e)
        rr = z.one("rr_ms", s, e) / 1000.0                           # seconds, O(1) scale
        if self.augment:
            if self.rng.rand() < 0.5:
                x = x + self.rng.randn(*x.shape).astype(np.float32) * 0.01
            if self.rng.rand() < 0.3:
                g = np.float32(1.0 + 0.1 * self.rng.randn())
                x = x * g
        return (torch.from_numpy(np.ascontiguousarray(x)),
                torch.from_numpy(y)[None, :],
                torch.from_numpy(pk)[None, :],
                torch.from_numpy(rr)[None, :])

def compute_norm(rec_dir, index, channels=CHANNELS, max_windows=4000, seed=0):
    # Per-channel mean/std computed on TRAIN WINDOWS ONLY. Computing them over the whole
    # corpus is a classic, invisible source of leakage.
    rng = np.random.RandomState(seed)
    idx = index if len(index) <= max_windows else index.iloc[
        rng.choice(len(index), max_windows, replace=False)]
    n = 0
    s1 = np.zeros(len(channels), np.float64)
    s2 = np.zeros(len(channels), np.float64)
    cache = {}
    rec_dir = Path(rec_dir)
    for _, r in idx.iterrows():
        rid = r["rec_id"]
        if rid not in cache:
            if len(cache) > 48:
                cache.pop(next(iter(cache)))
            cache[rid] = _Rec(rec_dir, rid)
        a, b = int(r["start"]), int(r["start"]) + WINDOW
        x = cache[rid].rows(list(channels), a, b).astype(np.float64)
        s1 += x.sum(1); s2 += (x * x).sum(1); n += x.shape[1]
    mean = s1 / max(n, 1)
    var = np.maximum(s2 / max(n, 1) - mean ** 2, 1e-12)
    return {"mean": mean.tolist(), "std": np.sqrt(var).tolist(),
            "n_samples": int(n), "channels": list(channels)}
"""
CRVS_METRICS_SRC = r"""
# crvs_metrics.py -- every metric the baseline reports, plus the ones it should have.
import numpy as np
from scipy import signal as ss
from scipy import stats as sstats

def _f(x):
    return np.nan_to_num(np.asarray(x, np.float64), nan=0.0, posinf=0.0, neginf=0.0)

def pearson(a, b):
    a, b = _f(a), _f(b)
    if a.std() < 1e-12 or b.std() < 1e-12:
        return 0.0
    return float(np.corrcoef(a, b)[0, 1])

def psd(x, fs=128, nperseg=256):
    f, p = ss.welch(_f(x), fs=fs, nperseg=min(nperseg, len(x)))
    return f, p

def seg_metrics(y, yhat, fs=128):
    # One window. Correlations are reported x100 to match the baseline's tables.
    y, yhat = _f(y), _f(yhat)
    mae = float(np.mean(np.abs(y - yhat)))
    mse = float(np.mean((y - yhat) ** 2))
    cct = 100.0 * pearson(y, yhat)
    _, py = psd(y, fs); _, ph = psd(yhat, fs)
    ccs = 100.0 * pearson(py, ph)
    rms = lambda v: float(np.sqrt(np.mean(np.asarray(v, np.float64) ** 2)))
    rr_t = rms(yhat - y) / (rms(y) + 1e-12)
    rr_s = rms(ph - py) / (rms(py) + 1e-12)
    return {"MAE": mae, "MSE": mse, "CC_temporal": cct, "CC_spectral": ccs,
            "RRMSE_temporal": rr_t, "RRMSE_spectral": rr_s,
            "R2": float(1.0 - np.sum((y - yhat) ** 2) / (np.sum((y - y.mean()) ** 2) + 1e-12))}

def detect_r_peaks(x, fs=128, refractory_s=0.25):
    x = _f(x)
    if len(x) < int(2 * fs):
        return np.array([], int)
    ny = fs / 2.0
    sos = ss.butter(4, [5.0 / ny, min(25.0, ny * 0.95) / ny], btype="band", output="sos")
    b = ss.sosfiltfilt(sos, x)
    e = np.convolve(np.diff(b, prepend=b[0]) ** 2,
                    np.ones(max(1, int(0.10 * fs))) / max(1, int(0.10 * fs)), "same")
    thr = np.percentile(e, 98) * 0.35
    pk, _ = ss.find_peaks(e, height=thr, distance=max(1, int(refractory_s * fs)))
    return pk

def hrv_from_peaks(pk, fs=128):
    out = {"n_peaks": int(len(pk)), "mean_rr_ms": np.nan, "sd_rr_ms": np.nan,
           "mean_hr_bpm": np.nan, "sd_hr_bpm": np.nan, "rmssd_ms": np.nan}
    if len(pk) < 4:
        return out
    rr = np.diff(np.asarray(pk, float)) / fs * 1000.0
    rr = rr[(rr > 300) & (rr < 2000)]
    if len(rr) < 3:
        return out
    hr = 60000.0 / rr
    out.update(mean_rr_ms=float(rr.mean()), sd_rr_ms=float(rr.std()),
               mean_hr_bpm=float(hr.mean()), sd_hr_bpm=float(hr.std()),
               rmssd_ms=float(np.sqrt(np.mean(np.diff(rr) ** 2))))
    return out

def peak_detection_scores(y, yhat, fs=128, tol_ms=100.0):
    # Match predicted R peaks to ground-truth peaks within a tolerance window.
    gt = detect_r_peaks(y, fs); pr = detect_r_peaks(yhat, fs)
    tol = tol_ms / 1000.0 * fs
    used = np.zeros(len(pr), bool)
    tp = 0
    errs = []
    for g in gt:
        if len(pr) == 0:
            break
        # float, not the int64 that find_peaks returns -- assigning np.inf into an
        # integer array raises OverflowError even when the mask selects nothing.
        d = np.abs(pr - g).astype(np.float64)
        d[used] = np.inf
        j = int(np.argmin(d))
        if d[j] <= tol:
            tp += 1; used[j] = True; errs.append((pr[j] - g) / fs * 1000.0)
    fp = int((~used).sum()); fn = int(len(gt) - tp)
    prec = tp / max(tp + fp, 1); rec = tp / max(tp + fn, 1)
    f1 = 2 * prec * rec / max(prec + rec, 1e-12)
    return {"TP": tp, "FP": fp, "FN": fn, "precision": prec, "recall": rec, "F1": f1,
            "accuracy": tp / max(tp + fp + fn, 1),
            "timing_err_ms_median": float(np.median(np.abs(errs))) if errs else np.nan,
            "timing_err_ms_iqr": float(np.subtract(*np.percentile(np.abs(errs), [75, 25])))
                                  if len(errs) > 3 else np.nan,
            "missed_rate": fn / max(len(gt), 1)}

def aggregate(rows):
    import pandas as pd
    df = pd.DataFrame(rows)
    out = {}
    for c in df.columns:
        if df[c].dtype.kind in "fi":
            out[c] = float(df[c].mean()); out[c + "_std"] = float(df[c].std())
    return out

def bland_altman(a, b):
    a, b = _f(a), _f(b)
    m = (a + b) / 2.0; d = a - b
    bias = float(d.mean()); sd = float(d.std())
    return {"mean": m, "diff": d, "bias": bias, "sd": sd,
            "loa_lo": bias - 1.96 * sd, "loa_hi": bias + 1.96 * sd}

def wilcoxon_holm(groups, better="higher"):
    # Pairwise Wilcoxon signed-rank across folds, Holm-corrected. groups: {name: [values]}
    import itertools
    names = list(groups)
    raw = []
    for a, b in itertools.combinations(names, 2):
        x, y = np.asarray(groups[a], float), np.asarray(groups[b], float)
        n = min(len(x), len(y))
        if n < 3 or np.allclose(x[:n], y[:n]):
            raw.append((a, b, np.nan)); continue
        try:
            p = float(sstats.wilcoxon(x[:n], y[:n]).pvalue)
        except Exception:
            p = np.nan
        raw.append((a, b, p))
    ps = [r[2] for r in raw]
    order = np.argsort([p if np.isfinite(p) else 1.0 for p in ps])
    m = len(ps); adj = [np.nan] * m; run = 0.0
    for k, i in enumerate(order):
        p = ps[i]
        if not np.isfinite(p):
            continue
        run = max(run, (m - k) * p)
        adj[i] = min(1.0, run)
    return [{"a": raw[i][0], "b": raw[i][1], "p": ps[i], "p_holm": adj[i]} for i in range(m)]
"""
for name, src in [("crvs_sync.py", CRVS_SYNC_SRC), ("crvs_data.py", CRVS_DATA_SRC),
                  ("crvs_metrics.py", CRVS_METRICS_SRC)]:
    (WORK / name).write_text(src)
    print(f"wrote {name}  ({len(src):,} chars)")
sys.path.insert(0, str(WORK))

In [ ]:
HF_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN loaded from Kaggle Secrets.")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")
    if not HF_TOKEN:
        raise RuntimeError("\n" + "="*74 +
            "\n  HF_TOKEN not found."
            "\n  Add-ons -> Secrets -> add HF_TOKEN (write scope) and attach it.\n" + "="*74)

from crvs_sync import HFSync
sync = HFSync(repo_id=CFG["DST_REPO"], local_dir=WORK, token=HF_TOKEN, repo_type="dataset",
              private=CFG["HF_PRIVATE"], run_id=CFG["RUN_ID"],
              push_interval_s=CFG["PUSH_INTERVAL_S"], max_req_hour=CFG["HF_MAX_REQ_HOUR"])
print("\ndestination:", sync.url, "(public)")

_MAJOR = {"f": False, "n": ""}
def MAJOR(name):
    _MAJOR["f"] = True; _MAJOR["n"] = name
def _hook(result=None):
    if _MAJOR["f"]:
        nm = _MAJOR["n"]; _MAJOR["f"] = False; _MAJOR["n"] = ""
        sync.stage_done(nm)
try:
    get_ipython().events.register("post_run_cell", _hook)
    print("post-run-cell push hook registered")
except Exception as e:
    print("hook unavailable:", e)

sync.pull(allow_patterns=["*.json", "*.jsonl", "*.parquet", "*.csv", "*.py", "*.md"])
STATE = sync.load_state({"done_recordings": [], "stages": {}})
print("resuming with", len(STATE["done_recordings"]), "recordings already built")
MAJOR("00_setup")

---
# 3 · Pull NB01's output

We need two things from the inventory repo: `inventory.csv` (to decide what to keep) and
`decimated/*.npz` (the 128 Hz channels). The download is ~400 MB and goes to scratch, not to
`/kaggle/working`, so it never counts against the 20 GB output budget.

If the decimated folder is missing — for instance if NB01 was run with `SAVE_DECIMATED = False`
— this cell says so and stops, rather than half-building a corpus.

In [ ]:
from huggingface_hub import snapshot_download
SRC = Path(SCRATCH) / "src"
t0 = time.time()
snapshot_download(CFG["SRC_REPO"], repo_type="dataset", token=HF_TOKEN,
                  local_dir=str(SRC),
                  allow_patterns=["inventory.csv", "verdict.json", "crosscheck.csv",
                                  "decimated/*.npz"])
print(f"downloaded in {time.time()-t0:.0f}s")

inv_path = SRC / "inventory.csv"
if not inv_path.exists():
    raise RuntimeError(f"inventory.csv not found in {CFG['SRC_REPO']} -- run NB01 first.")
INV = pd.read_csv(inv_path)
dec = sorted((SRC / "decimated").glob("*.npz"))
print(f"inventory rows : {len(INV)}")
print(f"decimated files: {len(dec)}")
if not dec:
    raise RuntimeError(
        "\n" + "="*74 +
        "\n  No decimated/*.npz found."
        "\n  Re-run NB01 with CFG['SAVE_DECIMATED'] = True, or point this notebook at the"
        "\n  raw .mat tree instead.\n" + "="*74)
print(f"total size     : {sum(p.stat().st_size for p in dec)/2**20:.0f} MB")
print("\ncolumns available:", len(INV.columns))

---
# 4 · Quality gate

Every exclusion is recorded with its reason, and the surviving corpus is compared against the
baseline's segment counts so we know exactly how much we gave up for cleanliness.

Two tiers. **Hard exclusions** are recordings the model cannot learn from: no radar channels, a
flatlined or unreadable ECG, NaNs, or — the one NB01 added — a `beat_coupling` below 1.3, meaning
the beat-triggered average of radar acceleration is indistinguishable from random triggers.
**Warnings** (clipping, receiver imbalance, an odd lag) are kept but carried through as columns,
so NB05 can test whether they correlate with reconstruction error.

In [ ]:
INV["scenario_canon"] = INV["scenario"].map(
    lambda s: ("Tilt-down" if "tiltdown" in str(s).lower().replace("_","").replace("-","")
               else "Tilt-up" if "tiltup" in str(s).lower().replace("_","").replace("-","")
               else "Valsalva" if "valsalva" in str(s).lower()
               else "Apnea" if "apnea" in str(s).lower() or "apnoea" in str(s).lower()
               else "Resting" if "rest" in str(s).lower() else str(s)))

flags_col = "quality_flags" if "quality_flags" in INV.columns else "flags"
INV[flags_col] = INV[flags_col].fillna("")

def reasons(r):
    out = []
    fl = set(str(r[flags_col]).split(";")) - {""}
    for f in CFG["EXCLUDE_FLAGS"]:
        if f in fl:
            out.append(f)
    bc = r.get("beat_coupling", np.nan)
    if pd.notna(bc) and bc < CFG["MIN_BEAT_COUPLING"]:
        out.append(f"beat_coupling<{CFG['MIN_BEAT_COUPLING']}")
    if pd.isna(bc):
        out.append("beat_coupling_missing")
    if r.get("duration_s", 0) < CFG["MIN_DURATION_S"]:
        out.append("too_short")
    return ";".join(out)

INV["exclude_reason"] = INV.apply(reasons, axis=1)
INV["keep"] = INV["exclude_reason"] == ""

print("=" * 84)
print("QUALITY GATE")
print("=" * 84)
summ = (INV.groupby(["scenario_canon", "keep"]).size().unstack(fill_value=0)
          .rename(columns={True: "kept", False: "dropped"}))
for c in ("kept", "dropped"):
    if c not in summ.columns:
        summ[c] = 0
summ["total"] = summ["kept"] + summ["dropped"]
summ["hours_kept"] = (INV[INV["keep"]].groupby("scenario_canon")["duration_s"].sum() / 3600).round(2)
print(summ.to_string())

print("\nexclusion reasons:")
from collections import Counter
cnt = Counter(x for s in INV.loc[~INV["keep"], "exclude_reason"] for x in s.split(";") if x)
if cnt:
    for k, v in cnt.most_common():
        print(f"  {k:<34} {v:>3} recording(s)")
else:
    print("  none -- every recording passed")

print(f"\nkept {int(INV['keep'].sum())}/{len(INV)} recordings, "
      f"{INV.loc[INV['keep'],'duration_s'].sum()/3600:.2f} h of "
      f"{INV['duration_s'].sum()/3600:.2f} h")
INV.to_csv(WORK / "inventory_gated.csv", index=False)
sync.log("quality_gate", kept=int(INV["keep"].sum()), total=len(INV))
MAJOR("01_quality_gate")

---
# 5 · Build the per-recording arrays

For every surviving recording we write one `.npz` holding the eight radar channels **unnormalised**
plus three targets:

- **`ecg_norm`** — the ECG z-scored, percentile-clipped and squashed to **[−1, 1]**. The baseline
  used [0, 1], which forces the network to spend capacity learning a DC offset it will never need.
  This is our one declared deviation from their pipeline, and NB03 re-runs their models under both.
- **`peak_map`** — a Gaussian bump (σ = 3 samples ≈ 23 ms) at every R peak. Target for the C4 peak head.
- **`rr_ms`** — per-sample instantaneous RR interval in **real milliseconds**, interpolated between
  beats. Target for the C4 cycle-length head, and the thing the baseline mislabelled as ms when it
  was actually samples.

Channels stay unnormalised on disk because normalisation statistics are computed **per fold, on
training windows only** (§7). Baking them in here would leak test statistics into training.

Resumable per recording.

In [ ]:
from crvs_data import (CHANNELS, ARRAY_ROWS, ROW, FS, WINDOW, HOP_TRAIN,
                       range_normalise, peak_heatmap, rr_curve)
from crvs_metrics import detect_r_peaks, hrv_from_peaks

keep = INV[INV["keep"]].copy()
if CFG["SMOKE_TEST"]:
    keep = keep.groupby("scenario_canon", group_keys=False).head(2)
    print(">>> SMOKE_TEST: only", len(keep), "recordings\n")

by_stem = {p.stem: p for p in dec}
done = set(STATE.get("done_recordings", []))
rows = STATE.get("rec_rows", [])
t0 = time.time(); built = 0; missing = []

for i, (_, r) in enumerate(keep.iterrows(), 1):
    rec_id = f"{r['subject']}__{r['scenario']}".replace("/", "_").replace(" ", "_")
    if rec_id in done:
        continue
    src = by_stem.get(rec_id)
    if src is None:
        cand = [s for s in by_stem if s.startswith(str(r["subject"]))
                and str(r["scenario"]).lower() in s.lower()]
        src = by_stem.get(cand[0]) if cand else None
    if src is None:
        missing.append(rec_id); continue

    z = np.load(src, allow_pickle=True)
    ch = {c: np.asarray(z[c], np.float32) for c in CHANNELS if c in z}
    if len(ch) != len(CHANNELS):
        missing.append(rec_id + " (channels)"); continue
    n = min(len(v) for v in ch.values())
    ecg = np.asarray(z["ecg1"], np.float32)[:n] if "ecg1" in z else None
    if ecg is None or len(ecg) < WINDOW:
        missing.append(rec_id + " (ecg)"); continue
    ch = {c: v[:n] for c, v in ch.items()}

    ecg_norm = range_normalise(ecg)
    peaks = detect_r_peaks(ecg_norm, FS)
    pk_map = peak_heatmap(n, peaks, CFG["PEAK_SIGMA"])
    rr = rr_curve(n, peaks, FS)
    hrv = hrv_from_peaks(peaks, FS)

    # One UNCOMPRESSED .npy of shape (11, n), row order = ARRAY_ROWS, plus a JSON sidecar.
    # It must not be .npz: np.load(..., mmap_mode="r") silently ignores mmap_mode on an
    # npz archive, so the Dataset would decompress all 11 arrays for every 1024-sample
    # window -- about 23 ms each, which would dominate T4 compute on Kaggle.
    stack = np.stack([ch[c] for c in CHANNELS] +
                     [ecg_norm, pk_map, rr.astype(np.float32)], 0).astype(np.float32)
    assert stack.shape == (len(ARRAY_ROWS), n), f"bad stack {stack.shape}"
    np.save(WORK / "recordings" / (rec_id + ".npy"), stack)
    (WORK / "recordings" / (rec_id + ".json")).write_text(json.dumps({
        "rec_id": rec_id, "fs": FS, "n": int(n),
        "subject": str(r["subject"]), "scenario": str(r["scenario_canon"]),
        "rows": ARRAY_ROWS, "r_peaks": [int(v) for v in peaks]}))

    rows.append({"rec_id": rec_id, "subject": str(r["subject"]),
                 "scenario_canon": str(r["scenario_canon"]), "n": int(n),
                 "duration_s": round(n / FS, 2), "n_rpeaks": int(len(peaks)),
                 "mean_hr_bpm": hrv["mean_hr_bpm"], "rmssd_ms": hrv["rmssd_ms"],
                 "beat_coupling": float(r.get("beat_coupling", np.nan)),
                 "warn_flags": ";".join(sorted(set(str(r[flags_col]).split(";")) &
                                               set(CFG["WARN_ONLY_FLAGS"])))})
    done.add(rec_id); built += 1
    if built % 10 == 0 or i == len(keep):
        STATE.update({"done_recordings": sorted(done), "rec_rows": rows})
        sync.save_state(STATE)
        el = time.time() - t0
        print(f"  [{len(done):>3}/{len(keep)}] {rec_id:<28} {n/FS:>7.1f}s  "
              f"HR={hrv['mean_hr_bpm'] if hrv['mean_hr_bpm']==hrv['mean_hr_bpm'] else 0:5.1f}  "
              f"ETA {el/max(built,1)*(len(keep)-len(done))/60:5.1f}m")
    del z, ch, ecg
    gc.collect()

STATE.update({"done_recordings": sorted(done), "rec_rows": rows})
sync.save_state(STATE)
RECS = pd.DataFrame(rows).drop_duplicates("rec_id").reset_index(drop=True)
RECS.to_csv(WORK / "recordings.csv", index=False)
print(f"\nbuilt {len(RECS)} recordings in {(time.time()-t0)/60:.1f} min")
if missing:
    print(f"WARNING: {len(missing)} recording(s) had no decimated file:", missing[:8])
    (WORK / "missing_recordings.json").write_text(json.dumps(missing, indent=2))
sync.log("recordings_built", n=len(RECS), missing=len(missing))
MAJOR("02_recordings")

---
# 6 · Folds and the window index

**Subject-wise 5-fold.** Subjects are sorted by how many scenarios they contributed and then
dealt round-robin into 5 groups, so each fold sees a comparable mix rather than one fold
accidentally collecting all the subjects who skipped Apnea. Fold *f* means: test = group *f*,
validation = group *(f+1) mod 5*, train = the other three. No subject is ever in two splits.

**LOSO.** Each subject gets an index; Experiment D iterates over all of them.

**The window index** is generated at 50 % overlap throughout, with a `no_overlap` column marking
windows that start on a 1024-sample boundary. Training uses every row; validation and test use
only `no_overlap` rows. One index, both behaviours, no chance of a stale second file drifting.

In [ ]:
rng = np.random.RandomState(CFG["SEED"])
subs = (RECS.groupby("subject")
            .agg(n_scen=("scenario_canon", "nunique"), secs=("duration_s", "sum"))
            .reset_index().sort_values(["n_scen", "secs"], ascending=False))
groups = {}
for k, s in enumerate(subs["subject"]):
    groups[s] = k % CFG["N_FOLDS"]                       # round-robin deal
loso = {s: i for i, s in enumerate(sorted(RECS["subject"].unique()))}

RECS["fold_group"] = RECS["subject"].map(groups)
RECS["loso_id"] = RECS["subject"].map(loso)
RECS.to_csv(WORK / "recordings.csv", index=False)

print("subjects per fold group:")
print(subs.assign(g=subs["subject"].map(groups)).groupby("g")["subject"].count().to_string())
print("\nrecordings per fold group x scenario:")
print(RECS.pivot_table(index="fold_group", columns="scenario_canon",
                       values="rec_id", aggfunc="count", fill_value=0).to_string())

widx = []
for _, r in RECS.iterrows():
    n = int(r["n"])
    if n < WINDOW:
        continue
    for st in range(0, n - WINDOW + 1, HOP_TRAIN):
        widx.append({"rec_id": r["rec_id"], "subject": r["subject"],
                     "scenario_canon": r["scenario_canon"], "start": st,
                     "no_overlap": (st % WINDOW) == 0,
                     "fold_group": int(r["fold_group"]), "loso_id": int(r["loso_id"])})
W = pd.DataFrame(widx)
W.to_parquet(WORK / "windows.parquet", index=False)

print(f"\n{len(W):,} windows total  ({int(W['no_overlap'].sum()):,} non-overlapping)")
tab = (W.groupby("scenario_canon")
        .agg(all_windows=("start", "size"), no_overlap=("no_overlap", "sum"),
             subjects=("subject", "nunique")).reset_index())
PAPER_SEG = {"Resting": 4702, "Valsalva": 6952, "Apnea": 1140}
tab["paper_segments"] = tab["scenario_canon"].map(PAPER_SEG)
print("\n" + tab.to_string(index=False))
rva = tab[tab["scenario_canon"].isin(["Resting", "Valsalva", "Apnea"])]
print(f"\nRVA overlapped windows kept: {int(rva['all_windows'].sum()):,}  "
      f"(baseline reported 12,794 before our quality gate)")
sync.log("windows", n=int(len(W)), no_overlap=int(W["no_overlap"].sum()))
MAJOR("03_windows")

---
# 7 · Per-fold normalisation statistics — train windows only

This is the cell that quietly decides whether the results are honest.

Channel statistics are computed **separately for every fold**, from **training windows only**.
If you compute one mean and standard deviation over the whole corpus and apply it everywhere,
information about the test subjects has entered training — invisibly, with no error message, and
in a way that inflates every metric you report. It is one of the most common silent mistakes in
biosignal papers.

Statistics are stored for each experiment × fold combination and loaded by NB03/NB04 at train time.

In [ ]:
from crvs_data import compute_norm

EXPERIMENTS = {
    "A_resting":  ["Resting"],
    "A_valsalva": ["Valsalva"],
    "A_apnea":    ["Apnea"],
    "B_rva":      ["Resting", "Valsalva", "Apnea"],
    "C_all5":     ["Resting", "Valsalva", "Apnea", "Tilt-up", "Tilt-down"],
}

def split_for(exp, fold, n_folds=None):
    n_folds = n_folds or CFG["N_FOLDS"]
    sub = W[W["scenario_canon"].isin(EXPERIMENTS[exp])]
    te_g = fold % n_folds
    va_g = (fold + 1) % n_folds
    tr = sub[~sub["fold_group"].isin([te_g, va_g])]
    va = sub[(sub["fold_group"] == va_g) & sub["no_overlap"]]
    te = sub[(sub["fold_group"] == te_g) & sub["no_overlap"]]
    return tr, va, te

NORM = {}
print(f"{'experiment':<12}{'fold':>5}{'train':>9}{'val':>8}{'test':>8}   train/val/test subjects")
print("-" * 84)
for exp in EXPERIMENTS:
    for f in range(CFG["N_FOLDS"]):
        tr, va, te = split_for(exp, f)
        if len(tr) == 0 or len(te) == 0:
            print(f"{exp:<12}{f:>5}   -- empty split, skipped"); continue
        st = compute_norm(WORK / "recordings", tr, seed=CFG["SEED"] + f)
        NORM[f"{exp}|{f}"] = st
        print(f"{exp:<12}{f:>5}{len(tr):>9,}{len(va):>8,}{len(te):>8,}   "
              f"{tr['subject'].nunique()}/{va['subject'].nunique()}/{te['subject'].nunique()}"
              f"   overlap={len(set(tr['subject']) & set(te['subject']))}")
        assert not (set(tr["subject"]) & set(te["subject"])), "SUBJECT LEAK train/test"
        assert not (set(va["subject"]) & set(te["subject"])), "SUBJECT LEAK val/test"

(WORK / "norm_stats.json").write_text(json.dumps(NORM, indent=2))
(WORK / "experiments.json").write_text(json.dumps(
    {"experiments": EXPERIMENTS, "n_folds": CFG["N_FOLDS"],
     "fold_groups": {str(k): int(v) for k, v in groups.items()},
     "loso_ids": {str(k): int(v) for k, v in loso.items()},
     "window": WINDOW, "hop_train": HOP_TRAIN, "fs": FS,
     "channels": CHANNELS}, indent=2))
print(f"\n{len(NORM)} normalisation sets written.  No subject appears in two splits anywhere.")
sync.log("norm_stats", n=len(NORM))
MAJOR("04_norm")

---
# 8 · Sanity figures

Four checks, because a corpus that is subtly wrong is expensive to discover in Week 4:

1. **A window as the model sees it** — all 8 input channels plus the three targets, time-aligned.
   If the ECG and the peak map are misaligned by even a few samples, it shows here.
2. **Beat-triggered ECG** — the ground-truth ECG averaged around its own detected peaks. Should
   look like a textbook QRS. If it does not, the peak detector is wrong and every target is wrong.
3. **Target distributions** — ECG amplitude, peak-map density, RR interval.
4. **Fold balance** — windows per fold per scenario, so no fold is starved.

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
STYLE = {"radar": "#0F7C82", "ecg": "#AF3A2C", "muted": "#5C6B71", "ink": "#10171B",
         "grid": "#D3DADB", "amber": "#8A6212"}
plt.rcParams.update({"figure.dpi": 130, "savefig.dpi": 160, "savefig.bbox": "tight",
                     "axes.grid": True, "grid.color": STYLE["grid"], "grid.linewidth": .6,
                     "axes.spines.top": False, "axes.spines.right": False,
                     "font.size": 8.5, "axes.titlesize": 10, "axes.titleweight": "bold"})
FIG = WORK / "figures"; FIG.mkdir(exist_ok=True)
def save(fig, nm):
    fig.savefig(FIG / nm); plt.close(fig); print("  wrote", nm)

ex = RECS.iloc[0]
z = np.load(WORK / "recordings" / (ex["rec_id"] + ".npy"), mmap_mode="r")
meta = json.loads((WORK / "recordings" / (ex["rec_id"] + ".json")).read_text())
s0 = min(int(20 * FS), max(0, int(meta["n"]) - WINDOW))
sl = slice(s0, s0 + WINDOW); t = np.arange(WINDOW) / FS

fig, axes = plt.subplots(11, 1, figsize=(11, 12), sharex=True)
for ax, c in zip(axes, CHANNELS):
    ax.plot(t, z[ROW[c], sl], lw=.7, color=STYLE["radar"])
    ax.set_ylabel(c, rotation=0, ha="right", va="center", fontsize=7.5)
    ax.tick_params(labelleft=False)
axes[8].plot(t, z[ROW["ecg_norm"], sl], lw=.9, color=STYLE["ecg"])
axes[8].set_ylabel("ecg_norm", rotation=0, ha="right", va="center", fontsize=7.5, color=STYLE["ecg"])
axes[9].plot(t, z[ROW["peak_map"], sl], lw=.9, color=STYLE["ink"])
axes[9].set_ylabel("peak_map", rotation=0, ha="right", va="center", fontsize=7.5)
axes[10].plot(t, z[ROW["rr_ms"], sl], lw=.9, color=STYLE["amber"])
axes[10].set_ylabel("rr_ms", rotation=0, ha="right", va="center", fontsize=7.5)
for a in axes[8:]:
    a.tick_params(labelleft=False)
axes[-1].set_xlabel("seconds")
axes[0].set_title(f"{ex['rec_id']} — one 8 s window exactly as the model receives it", loc="left")
save(fig, "nb02_fig1_window.png")

fig, axes = plt.subplots(1, 3, figsize=(12, 3))
pk = np.asarray(meta["r_peaks"], int); e = np.asarray(z[ROW["ecg_norm"]])
pre, post = int(.25 * FS), int(.45 * FS)
seg = [e[p-pre:p+post] for p in pk if p-pre >= 0 and p+post < len(e)]
if seg:
    S = np.stack(seg); tt = np.arange(-pre, post) / FS
    for s_ in S[:200]:
        axes[0].plot(tt, s_, lw=.3, color=STYLE["muted"], alpha=.25)
    axes[0].plot(tt, S.mean(0), lw=2, color=STYLE["ecg"])
    axes[0].axvline(0, color=STYLE["ink"], ls="--", lw=1)
    axes[0].set_title(f"beat-triggered ECG (n={len(S)})"); axes[0].set_xlabel("s from R peak")
axes[1].hist(RECS["mean_hr_bpm"].dropna(), bins=20, color=STYLE["ecg"], edgecolor="white")
axes[1].set_title("mean HR per recording"); axes[1].set_xlabel("bpm")
axes[2].hist(RECS["rmssd_ms"].dropna(), bins=20, color=STYLE["radar"], edgecolor="white")
axes[2].set_title("RMSSD per recording"); axes[2].set_xlabel("ms (real)")
fig.tight_layout(); save(fig, "nb02_fig2_targets.png")

fig, ax = plt.subplots(figsize=(9, 3.2))
pv = W.pivot_table(index="fold_group", columns="scenario_canon", values="start",
                   aggfunc="size", fill_value=0)
pv.plot(kind="bar", stacked=True, ax=ax, width=.75,
        color=[STYLE["radar"], STYLE["ecg"], STYLE["amber"], STYLE["muted"], "#9BB8BA"],
        edgecolor="white")
ax.set_ylabel("windows"); ax.set_xlabel("fold group")
ax.set_title("Window balance across subject-wise folds", loc="left")
ax.legend(frameon=False, ncol=5, fontsize=7)
save(fig, "nb02_fig3_folds.png")
MAJOR("05_figures")

---
# 9 · Dataset card and final push

The processed repo is what NB03–NB05 consume, so its card doubles as the contract between the
notebooks: channel order, window length, fold semantics, and the leakage rules.

In [ ]:
now = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M UTC")
card = f"""---
license: cc-by-4.0
pretty_name: CR-RVS Radar-to-ECG Training Corpus
tags: [radar, ecg, biosignals, contactless-monitoring, vital-signs]
---

# CR-RVS Radar-to-ECG — training corpus

Windowed, fold-assigned training corpus for **CardioMamba-Net**, derived from the CR-RVS dataset
(Schellenberger et al., *Sci Data* 7:291, 2020) via `01_verify_and_download` and
`02_preprocess_to_hf`. Built {now}, run `{CFG['RUN_ID']}`.

## Contents

| Path | What |
|---|---|
| `recordings/<subject>__<scenario>.npy` | Uncompressed `(11, n)` float32 at 128 Hz — 8 radar channels + 3 targets, row order in `rows` |
| `recordings/<subject>__<scenario>.json` | Per-recording metadata: `n`, `fs`, `subject`, `scenario`, `rows`, `r_peaks` |
| `windows.parquet` | One row per window: `rec_id, subject, scenario_canon, start, no_overlap, fold_group, loso_id` |
| `recordings.csv` | Per-recording metadata, HR/HRV, beat coupling, warning flags |
| `inventory_gated.csv` | Full NB01 inventory plus `keep` / `exclude_reason` |
| `norm_stats.json` | Per experiment x fold channel mean/std, **train windows only** |
| `experiments.json` | Experiment definitions, fold groups, LOSO ids, channel order |
| `crvs_sync.py`, `crvs_data.py`, `crvs_metrics.py` | Shared library used by NB03-NB05 |
| `figures/` | Sanity figures |

## Arrays in each recording

Row order of the `(11, n)` array: `I, Q, phi, dy, vel, acc, amp, cardiac` — the 8
physics-informed input channels, **unnormalised** — then `ecg_norm` (target, [-1,1]),
`peak_map` (Gaussian R-peak heatmap, sigma = 3 samples), `rr_ms` (per-sample RR interval in
**real milliseconds**). R-peak indices are in the JSON sidecar.

Stored uncompressed on purpose: `np.load(..., mmap_mode="r")` silently ignores `mmap_mode`
on an `.npz`, so a compressed archive would force a full decompression of all 11 arrays for
every 1024-sample window.

## Conventions

- 128 Hz, 1024-sample (8 s) windows, frozen to Chowdhury et al. 2024 section 2.3.
- Train windows overlap 50 %; **validation and test windows do not overlap** (`no_overlap`).
- Splits are **always by subject**. Fold f: test = group f, val = group (f+1) mod {CFG['N_FOLDS']}, train = rest.
- Normalisation statistics are per fold and computed on **training windows only**.
- ECG target is [-1, 1], not [0, 1] as in the baseline; this is a declared deviation.

## Quality gate

Recordings are excluded when the radar cannot be shown to see the heartbeat
(`beat_coupling < {CFG['MIN_BEAT_COUPLING']}`), when the ECG is flat or unreadable, when radar
channels are missing, or when the recording is under {CFG['MIN_DURATION_S']:.0f} s. Every
exclusion and its reason is in `inventory_gated.csv`.

## Cite

Schellenberger et al., *Scientific Data* 7:291 (2020), doi:10.1038/s41597-020-00629-5 ·
Chowdhury et al., *Computers in Biology and Medicine* 176:108555 (2024),
doi:10.1016/j.compbiomed.2024.108555
"""
(WORK / "README.md").write_text(card)

report = [f"# NB02 report — {now}\n",
          f"- recordings built: **{len(RECS)}**",
          f"- windows: **{len(W):,}** ({int(W['no_overlap'].sum()):,} non-overlapping)",
          f"- subjects: **{RECS['subject'].nunique()}**",
          f"- hours kept: **{RECS['duration_s'].sum()/3600:.2f} h**",
          f"- normalisation sets: **{len(NORM)}**\n",
          "## Per scenario\n", tab.to_markdown(index=False), "\n## Quality gate\n",
          summ.to_markdown()]
(WORK / "report.md").write_text("\n".join(report))

sizes = {str(p.relative_to(WORK)): p.stat().st_size for p in WORK.rglob("*") if p.is_file()}
print(f"pushing {len(sizes)} files, {sum(sizes.values())/2**20:.0f} MB ...")
ok = sync.flush(final=True, msg=f"{CFG['RUN_ID']} complete — {len(RECS)} recordings, {len(W)} windows")
print("\n" + "=" * 76)
print("  DONE" if ok else "  DONE (final push reported a problem — see history.jsonl)")
print("=" * 76)
print(f"  repo       : {sync.url}")
print(f"  recordings : {len(RECS)}   windows: {len(W):,}   subjects: {RECS['subject'].nunique()}")
print(f"  payload    : {sum(sizes.values())/2**20:.0f} MB in {len(sizes)} files")
print(f"  pushes     : {sync._pushes}  failures: {sync._failures}")
print("=" * 76)
print("\n  next: 03_baselines.ipynb  (accelerator: GPU T4 x2)")

---
# 10 · Troubleshooting

**`inventory.csv not found`** — NB01 has not completed, or it pushed to a different repo. Check
`CFG["SRC_REPO"]` matches NB01's `CFG["HF_REPO"]`.

**`No decimated/*.npz found`** — NB01 ran with `SAVE_DECIMATED = False`. Re-run NB01 with it on;
it resumes and only needs to write the decimated files.

**Out of disk** — the download goes to `/kaggle/temp`, the output to `/kaggle/working` (20 GB cap).
The recordings folder is ~400 MB, so this should not happen. If it does, run with `SMOKE_TEST = True`
first to confirm the pipeline, then re-run.

**`SUBJECT LEAK` assertion** — a real bug, not a warning. It means fold assignment produced an
overlapping subject between splits. Do not bypass it; tell me and I will fix the fold logic.

**Recordings missing from `decimated/`** — listed in `missing_recordings.json`. Usually a
subject/scenario naming mismatch between the inventory and the decimated filenames. The cell
already tries a fuzzy fallback; if many are missing, send me that file.